In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("No GPU detected.")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2"
)

print("Data directory exists:", DATA_DIR.exists())

for filename in [
    "train_scaled_sequences.npz",
    "validation_scaled_sequences.npz",
    "test_scaled_sequences.npz",
]:
    file_path = DATA_DIR / filename

    if file_path.exists():
        print(f"✓ {filename} — {file_path.stat().st_size / (1024**2):.2f} MB")
    else:
        print(f"✗ Missing: {filename}")

Data directory exists: True
✓ train_scaled_sequences.npz — 14.51 MB
✓ validation_scaled_sequences.npz — 3.06 MB
✓ test_scaled_sequences.npz — 3.07 MB


In [ ]:
!git clone --branch feature/agent2-data-pipeline \
https://github.com/kameldemri/multi-agent-disinformation-detection.git

Cloning into 'multi-agent-disinformation-detection'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 151 (delta 55), reused 121 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 580.49 KiB | 11.16 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [ ]:
import sys

PROJECT_DIR = "/content/multi-agent-disinformation-detection"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Project root added to Python path.")

Project root added to Python path.


In [ ]:
from agents.agent2_rnn.src.dataset import create_dataloader
from agents.agent2_rnn.src.model import AccountBiGRU
from agents.agent2_rnn.src.train import train_model, CONFIG

print("All imports successful!")
print("\nTraining configuration:")

for key, value in CONFIG.items():
    print(f"{key}: {value}")

All imports successful!

Training configuration:
input_size: 5
hidden_size: 64
num_layers: 1
dropout: 0.3
batch_size: 16
bucket_size_multiplier: 10
learning_rate: 0.001
weight_decay: 0.01
max_epochs: 30
patience: 5
gradient_clip: 1.0
threshold: 0.5
seed: 42


In [ ]:
from pathlib import Path
from agents.agent2_rnn.src import train

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/multi-agent-disinformation-detection/checkpoints/agent2"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Redirect the training controller's output directory
train.CHECKPOINT_DIR = OUTPUT_DIR

print("Checkpoint directory:", train.CHECKPOINT_DIR)
print("Directory exists:", train.CHECKPOINT_DIR.exists())

Checkpoint directory: /content/drive/MyDrive/multi-agent-disinformation-detection/checkpoints/agent2
Directory exists: True


In [ ]:
from pathlib import Path
from agents.agent2_rnn.src import preprocess

# Actual location of the processed datasets in Google Drive
DATA_DIR = Path(
    "/content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2"
)

# Update the directory used by the dataset pipeline
preprocess.PROCESSED_DATA_DIR = DATA_DIR

print("Updated PROCESSED_DATA_DIR:", preprocess.PROCESSED_DATA_DIR)
print("Directory exists:", preprocess.PROCESSED_DATA_DIR.exists())

for split in ["train", "validation", "test"]:
    path = preprocess.PROCESSED_DATA_DIR / f"{split}_scaled_sequences.npz"
    print(f"{split}: {path.exists()} | {path}")

Updated PROCESSED_DATA_DIR: /content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2
Directory exists: True
train: True | /content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2/train_scaled_sequences.npz
validation: True | /content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2/validation_scaled_sequences.npz
test: True | /content/drive/MyDrive/multi-agent-disinformation-detection/data/agent2/test_scaled_sequences.npz


In [ ]:
from agents.agent2_rnn.src.dataset import AccountSequenceDataset

train_dataset = AccountSequenceDataset("train")
validation_dataset = AccountSequenceDataset("validation")

Loading train dataset...
  Accounts: 1,451
  Total timesteps: 3,132,339
  Min length: 13
  Max length: 3,250
Loading validation dataset...
  Accounts: 311
  Total timesteps: 656,018
  Min length: 3
  Max length: 3,249


In [ ]:
print("TRAIN DATASET")
print("Number of accounts:", len(train_dataset))

print("\nVALIDATION DATASET")
print("Number of accounts:", len(validation_dataset))

TRAIN DATASET
Number of accounts: 1451

VALIDATION DATASET
Number of accounts: 311


In [ ]:
from agents.agent2_rnn.src.dataset import create_dataloader
from agents.agent2_rnn.src.train import CONFIG

train_loader = create_dataloader(
    split="train",
    batch_size=CONFIG["batch_size"],
    bucket_size_multiplier=CONFIG["bucket_size_multiplier"],
    seed=CONFIG["seed"],
)

validation_loader = create_dataloader(
    split="validation",
    batch_size=CONFIG["batch_size"],
    bucket_size_multiplier=CONFIG["bucket_size_multiplier"],
    seed=CONFIG["seed"],
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

Loading train dataset...
  Accounts: 1,451
  Total timesteps: 3,132,339
  Min length: 13
  Max length: 3,250
Loading validation dataset...
  Accounts: 311
  Total timesteps: 656,018
  Min length: 3
  Max length: 3,249
Training batches: 91
Validation batches: 20


In [ ]:
# Retrieve one training batch
sequences, labels, lengths = next(iter(train_loader))

print("Sequences shape:", sequences.shape)
print("Labels shape:", labels.shape)
print("Lengths shape:", lengths.shape)

print("\nSequence data type:", sequences.dtype)
print("Labels data type:", labels.dtype)
print("Lengths data type:", lengths.dtype)

print("\nSequence lengths:", lengths.tolist())
print("Are lengths sorted descending?",
      bool((lengths[:-1] >= lengths[1:]).all()))

print("\nUnique labels:", labels.unique().tolist())
print("All sequence values finite:", bool(sequences.isfinite().all()))

Sequences shape: torch.Size([16, 3204, 5])
Labels shape: torch.Size([16])
Lengths shape: torch.Size([16])

Sequence data type: torch.float32
Labels data type: torch.float32
Lengths data type: torch.int64

Sequence lengths: [3204, 3204, 3204, 3202, 3200, 3199, 3198, 3197, 3195, 3189, 3186, 3186, 3185, 3184, 3184, 3182]
Are lengths sorted descending? True

Unique labels: [0.0, 1.0]
All sequence values finite: True


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: cuda
GPU: Tesla T4


In [ ]:
from agents.agent2_rnn.src.model import AccountBiGRU

model = AccountBiGRU(
    input_size=CONFIG["input_size"],
    hidden_size=CONFIG["hidden_size"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"],
).to(device)

print(model)

AccountBiGRU(
  (gru): GRU(5, 64, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=128, out_features=1, bias=True)
)


In [ ]:
total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"\nTrainable parameters: {total_params:,}")


Trainable parameters: 27,393


In [ ]:
from agents.agent2_rnn.src.train import train_model, set_seed

# Reproducibility
set_seed(CONFIG["seed"])

# Reset the sampler before training
train_loader.batch_sampler.epoch = 0

# Launch training
history = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    device=device,
)

In [ ]:
import torch

best_checkpoint_path = OUTPUT_DIR / "best_bigru.pt"

print("Checkpoint exists:", best_checkpoint_path.exists())

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
    weights_only=True,
)

# Restore the best model weights
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("\nBest checkpoint successfully restored!")
print("Best epoch:", checkpoint["epoch"])
print("Validation metrics:", checkpoint["validation_metrics"])

Checkpoint exists: True

Best checkpoint successfully restored!
Best epoch: 8
Validation metrics: {'loss': 0.08698535534490343, 'accuracy': 0.9678456591639871, 'precision': 1.0, 'recall': 0.9328859060402684, 'f1': 0.9652777777777778, 'roc_auc': 0.9944071588366891}


In [ ]:
import pandas as pd

history_path = train.CHECKPOINT_DIR / "training_history.csv"

print("History file exists:", history_path.exists())

if history_path.exists():
    history_df = pd.read_csv(history_path)
    display(history_df)

History file exists: True


,epoch,train_loss,val_loss,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc
0,1,0.432579,0.204869,0.932476,0.921053,0.939597,0.930233,0.977877
1,2,0.184524,0.190991,0.932476,0.910256,0.953020,0.931148,0.980529
2,3,0.157569,0.165043,0.945338,0.940000,0.946309,0.943144,0.982351
3,4,0.128123,0.158310,0.951768,0.952703,0.946309,0.949495,0.982973
4,5,0.115827,0.118334,0.964630,0.979167,0.946309,0.962457,0.987489
5,6,0.105001,0.126028,0.964630,0.985915,0.939597,0.962199,0.988524
6,7,0.104317,0.091926,0.964630,0.972603,0.953020,0.962712,0.992336
7,8,0.074628,0.086985,0.967846,1.000000,0.932886,0.965278,0.994407
8,9,0.079180,0.114736,0.958199,1.000000,0.912752,0.954386,0.993330
9,10,0.059512,0.072368,0.980707,1.000000,0.959732,0.979452,0.993620
